# Brazilian E-Commerce Public Dataset by Olist

https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data


Este é um conjunto de dados público brasileiro de e-commerce com pedidos realizados na Olist Store . O conjunto de dados contém informações de 100 mil pedidos feitos entre 2016 e 2018 em diversos marketplaces no Brasil. Suas funcionalidades permitem visualizar um pedido sob múltiplas perspectivas: desde o status do pedido, preço, pagamento e desempenho do frete até a localização do cliente, atributos do produto e, por fim, avaliações escritas pelos clientes. Também disponibilizamos um conjunto de dados de geolocalização que relaciona CEPs brasileiros a coordenadas de latitude e longitude.

Estes são dados comerciais reais, que foram anonimizados, e as referências às empresas e parceiros no texto da análise foram substituídas pelos nomes das grandes casas de Game of Thrones.




In [0]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [0]:
base_path = Path("../data")

olist_customers = pd.read_csv(
    base_path / "olist_customers_dataset.csv"
)

olist_orders = pd.read_csv(
    base_path / "olist_orders_dataset.csv"
)

olist_order_items = pd.read_csv(
    base_path / "olist_order_items_dataset.csv"
)

olist_geolocation = pd.read_csv(
    base_path / "olist_geolocation_dataset.csv"
)

olist_order_payments = pd.read_csv(
    base_path / "olist_order_payments_dataset.csv"
)

olist_order_reviews = pd.read_csv(
    base_path / "olist_order_reviews_dataset.csv"
)

olist_products = pd.read_csv(
    base_path / "olist_products_dataset.csv"
)

olist_sellers = pd.read_csv(
    base_path / "olist_sellers_dataset.csv"
)

olist_product_name = pd.read_csv(
    base_path / "product_category_name_translation.csv"
)


## Inventário dos Dados

In [0]:
dataframes = {
    "customers": olist_customers,
    "geolocation": olist_geolocation,
    "order_items": olist_order_items,
    "order_payments": olist_order_payments,
    "order_reviews": olist_order_reviews,
    "orders": olist_orders,
    "products": olist_products,
    "sellers": olist_sellers,
    "product_name": olist_product_name,
}

In [0]:
inventario = pd.DataFrame([
    {
        "tabela": nome,
        "linhas": df.shape[0],
        "colunas": df.shape[1]
    }
    for nome, df in dataframes.items()
])

inventario

In [0]:

for nome, df in dataframes.items():
    print(f"\n{'=' * 50}")
    print(f"TABELA: {nome}")
    print(df.info())


## Qualidade

In [0]:

# Diagnóstico de valores ausentes por tabela

relatorio_nulos = []

for nome, df in dataframes.items():
    for coluna in df.columns:
        qtd_nulos = df[coluna].isna().sum()

        relatorio_nulos.append({
            "tabela": nome,
            "coluna": coluna,
            "total_linhas": len(df),
            "qtd_nulos": qtd_nulos,
            "percentual_nulos": round(
                (qtd_nulos / len(df)) * 100, 2
            ) if len(df) > 0 else 0
        })

df_nulos = pd.DataFrame(relatorio_nulos)

# Exibir somente colunas com valores ausentes
df_nulos = (
    df_nulos[df_nulos["qtd_nulos"] > 0]
    .sort_values(
        by=["percentual_nulos", "qtd_nulos"],
        ascending=False
    )
    .reset_index(drop=True)
)

df_nulos


In [0]:

# Estatísticas descritivas das colunas numéricas

for nome, df in dataframes.items():
    print(f"\n{'=' * 60}")
    print(f"TABELA: {nome}")
    display(df.describe().T)


## Análise

In [0]:

# Visão analítica de pedidos e entregas

pedidos_analise = (
    dataframes["orders"]
    .merge(
        dataframes["customers"],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
)

print("Linhas na tabela original:", len(dataframes["orders"]))
print("Linhas na tabela analítica:", len(pedidos_analise))
print("Pedidos únicos:", pedidos_analise["order_id"].nunique())


In [0]:

plt.rcParams.update({
    "figure.figsize": (10, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

orders = dataframes["orders"].copy()

cols_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in cols_data:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

orders["mes_compra"] = orders["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()

itens_pedido = (
    dataframes["order_items"]
    .groupby("order_id")
    .agg(
        receita_produtos=("price", "sum"),
        frete=("freight_value", "sum"),
        qtd_itens=("order_item_id", "count"),
    )
    .reset_index()
)

orders = (
    orders
    .merge(dataframes["customers"], on="customer_id", how="left", validate="many_to_one")
    .merge(itens_pedido, on="order_id", how="left", validate="one_to_one")
)

validos = orders[~orders["order_status"].isin(["canceled", "unavailable"])].copy()

print(f"Pedidos totais: {len(orders):,}")
print(f"Pedidos válidos (sem cancelados/indisponíveis): {len(validos):,}")
print(f"Pedidos sem nenhum item em order_items: {orders['receita_produtos'].isna().sum():,}")
print(f"Período: {orders['order_purchase_timestamp'].min():%Y-%m-%d} a {orders['order_purchase_timestamp'].max():%Y-%m-%d}")

## 1. Evolução de pedidos e receita

Receita aqui = soma de `price` dos itens (sem frete), considerando só pedidos válidos.

In [0]:
mensal = (
    validos
    .groupby("mes_compra")
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("receita_produtos", "sum"),
        ticket_medio=("receita_produtos", "mean"),
    )
)

display(mensal.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(mensal.index, mensal["pedidos"], marker="o")
axes[0].set_title("Pedidos por mês")
axes[0].set_ylabel("Pedidos")

axes[1].plot(mensal.index, mensal["receita"] / 1e3, marker="o", color="tab:green")
axes[1].set_title("Receita de produtos por mês (R$ mil)")
axes[1].set_ylabel("R$ mil")

for ax in axes:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [0]:
# Crescimento mês a mês (desconsiderando meses com poucos pedidos, que distorcem o %)
mensal_ok = mensal[mensal["pedidos"] >= 100].copy()
mensal_ok["cresc_pedidos_pct"] = mensal_ok["pedidos"].pct_change() * 100
mensal_ok["cresc_receita_pct"] = mensal_ok["receita"].pct_change() * 100
display(mensal_ok[["pedidos", "cresc_pedidos_pct", "receita", "cresc_receita_pct"]].round(1))

### Sazonalidade: dia da semana e hora

In [0]:
dias = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]

por_dia = validos["order_purchase_timestamp"].dt.dayofweek.value_counts().sort_index()
por_hora = validos["order_purchase_timestamp"].dt.hour.value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar([dias[i] for i in por_dia.index], por_dia.values)
axes[0].set_title("Pedidos por dia da semana")

axes[1].bar(por_hora.index, por_hora.values)
axes[1].set_title("Pedidos por hora do dia")
axes[1].set_xlabel("Hora")

plt.tight_layout()
plt.show()

## 2. Status dos pedidos

Também testamos a hipótese levantada na etapa de qualidade: os ~3% de `order_delivered_customer_date` nulos correspondem a pedidos que não chegaram ao cliente (cancelados, em trânsito etc.)?

In [0]:
status = orders["order_status"].value_counts().rename("pedidos").to_frame()
status["pct"] = (status["pedidos"] / status["pedidos"].sum() * 100).round(2)
display(status)

# Nulos em data de entrega por status
nulos_entrega = (
    orders
    .assign(sem_data_entrega=orders["order_delivered_customer_date"].isna())
    .groupby("order_status")["sem_data_entrega"]
    .agg(sem_data="sum", total="count")
)
nulos_entrega["pct_sem_data"] = (nulos_entrega["sem_data"] / nulos_entrega["total"] * 100).round(1)
display(nulos_entrega.sort_values("sem_data", ascending=False))

# Pedidos "delivered" sem data de entrega seriam um problema de qualidade real
inconsistentes = orders[(orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isna()]
print(f"Pedidos 'delivered' sem data de entrega: {len(inconsistentes)}")

## 3. Desempenho de entrega

Usamos apenas pedidos `delivered` com data de entrega preenchida.

- **dias_entrega**: compra → entrega ao cliente.
- **dias_atraso**: entrega − data estimada (positivo = atrasou). A comparação é feita por **dia** (`normalize`), porque a data estimada vem sem horário; sem isso, uma entrega no dia previsto à tarde seria contada como atraso.

In [0]:
entregues = orders[
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].notna()
].copy()

entregues["dias_entrega"] = (
    entregues["order_delivered_customer_date"] - entregues["order_purchase_timestamp"]
).dt.total_seconds() / 86400

entregues["dias_previstos"] = (
    entregues["order_estimated_delivery_date"] - entregues["order_purchase_timestamp"]
).dt.total_seconds() / 86400

entregues["dias_atraso"] = (
    entregues["order_delivered_customer_date"].dt.normalize()
    - entregues["order_estimated_delivery_date"].dt.normalize()
).dt.days

entregues["atrasado"] = entregues["dias_atraso"] > 0

print(f"Pedidos entregues analisados: {len(entregues):,}")
print(f"% entregues com atraso: {entregues['atrasado'].mean() * 100:.1f}%")
print()
display(entregues[["dias_entrega", "dias_previstos", "dias_atraso"]].describe().round(1))

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(entregues["dias_entrega"].clip(upper=60), bins=60, color="tab:blue")
axes[0].axvline(entregues["dias_entrega"].median(), color="red", ls="--",
                label=f"mediana: {entregues['dias_entrega'].median():.1f} dias")
axes[0].set_title("Tempo de entrega (dias) — cortado em 60")
axes[0].legend()

axes[1].hist(entregues["dias_atraso"].clip(-30, 30), bins=60, color="tab:orange")
axes[1].axvline(0, color="black", lw=1)
axes[1].set_title("Entrega − previsão (dias) — cortado em ±30\n(negativo = adiantou)")

plt.tight_layout()
plt.show()

In [0]:
# Entrega por estado do cliente
por_uf = (
    entregues
    .groupby("customer_state")
    .agg(
        pedidos=("order_id", "count"),
        dias_medios=("dias_entrega", "mean"),
        pct_atrasados=("atrasado", "mean"),
    )
    .assign(pct_atrasados=lambda d: d["pct_atrasados"] * 100)
    .sort_values("dias_medios")
)
display(por_uf.round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].barh(por_uf.index, por_uf["dias_medios"], color="tab:blue")
axes[0].set_title("Tempo médio de entrega (dias) por UF")

p = por_uf.sort_values("pct_atrasados")
axes[1].barh(p.index, p["pct_atrasados"], color="tab:red")
axes[1].set_title("% de pedidos entregues com atraso por UF")

plt.tight_layout()
plt.show()

In [0]:
# Tempo de entrega x frete: estados distantes pagam mais frete?
frete_uf = (
    entregues
    .assign(frete_pct=lambda d: d["frete"] / (d["receita_produtos"] + d["frete"]) * 100)
    .groupby("customer_state")
    .agg(frete_medio=("frete", "mean"), frete_pct_medio=("frete_pct", "mean"))
    .join(por_uf[["dias_medios"]])
)
display(frete_uf.sort_values("frete_medio", ascending=False).round(1))
print("Correlação dias de entrega x frete médio entre UFs:",
      round(frete_uf["dias_medios"].corr(frete_uf["frete_medio"]), 2))

## 4. Categorias de produto

Produtos sem categoria (610 no dataset) entram como `desconhecida`. Algumas categorias não têm tradução na tabela `product_category_name_translation`; nesses casos mantemos o nome em português.

In [0]:
produtos = (
    dataframes["products"][["product_id", "product_category_name"]]
    .merge(dataframes["product_name"], on="product_category_name", how="left")
)

produtos["categoria"] = (
    produtos["product_category_name_english"]
    .fillna(produtos["product_category_name"])
    .fillna("desconhecida")
)

itens = (
    dataframes["order_items"]
    .merge(produtos[["product_id", "categoria"]], on="product_id", how="left", validate="many_to_one")
)
itens = itens[itens["order_id"].isin(validos["order_id"])]

categorias = (
    itens
    .groupby("categoria")
    .agg(
        receita=("price", "sum"),
        itens=("order_id", "count"),
        pedidos=("order_id", "nunique"),
        ticket_medio=("price", "mean"),
    )
    .sort_values("receita", ascending=False)
)
categorias["pct_receita"] = categorias["receita"] / categorias["receita"].sum() * 100
categorias["pct_acumulado"] = categorias["pct_receita"].cumsum()

print(f"Categorias distintas: {len(categorias)}")
display(categorias.head(15).round(1))

top = categorias.head(10).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(top.index, top["receita"] / 1e3, color="tab:green")
axes[0].set_title("Top 10 categorias por receita (R$ mil)")

axes[1].barh(top.index, top["ticket_medio"], color="tab:purple")
axes[1].set_title("Preço médio dos itens nessas categorias (R$)")

plt.tight_layout()
plt.show()

print(f"As 10 maiores categorias concentram {categorias['pct_receita'].head(10).sum():.1f}% da receita.")

## 5. Pagamentos

Um pedido pode ter várias linhas em `order_payments` (várias formas de pagamento ou parcelas de voucher). Por isso, ao contar pedidos por tipo, a soma passa do total de pedidos.

In [0]:
pag = dataframes["order_payments"]

por_tipo = (
    pag
    .groupby("payment_type")
    .agg(
        pedidos=("order_id", "nunique"),
        valor_total=("payment_value", "sum"),
        valor_medio=("payment_value", "mean"),
    )
    .sort_values("valor_total", ascending=False)
)
por_tipo["pct_valor"] = (por_tipo["valor_total"] / por_tipo["valor_total"].sum() * 100).round(1)
display(por_tipo.round(1))

multi = (pag.groupby("order_id")["payment_type"].nunique() > 1).mean() * 100
print(f"Pedidos pagos com mais de uma forma de pagamento: {multi:.1f}%")

In [0]:
# Parcelamento no cartão de crédito (máximo de parcelas por pedido)
parcelas = (
    pag[pag["payment_type"] == "credit_card"]
    .groupby("order_id")["payment_installments"]
    .max()
    .clip(upper=10)
)
dist = parcelas.value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].bar(dist.index.astype(str), dist.values)
axes[0].set_title("Pedidos no cartão por nº de parcelas (10 = 10 ou mais)")
axes[0].set_xlabel("Parcelas")

valor_pedido = pag.groupby("order_id")["payment_value"].sum()
axes[1].hist(valor_pedido.clip(upper=1000), bins=50, color="tab:green")
axes[1].set_title("Valor pago por pedido (R$) — cortado em 1.000")

plt.tight_layout()
plt.show()

print(f"Pedidos no cartão em 1x: {(parcelas == 1).mean() * 100:.1f}%")
print(f"Pedidos no cartão em 6x ou mais: {(parcelas >= 6).mean() * 100:.1f}%")

## 6. Avaliações dos clientes

Alguns pedidos têm mais de uma avaliação. Para não contar o mesmo pedido duas vezes, mantemos a **mais recente** de cada um.

In [0]:
av = dataframes["order_reviews"].copy()
av["review_answer_timestamp"] = pd.to_datetime(av["review_answer_timestamp"], errors="coerce")

print(f"Avaliações: {len(av):,} | pedidos distintos: {av['order_id'].nunique():,} "
      f"| pedidos com mais de 1 avaliação: {av['order_id'].duplicated().sum():,}")

av_pedido = (
    av.sort_values("review_answer_timestamp")
      .drop_duplicates("order_id", keep="last")
      [["order_id", "review_score"]]
)

dist_notas = av_pedido["review_score"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(dist_notas.index.astype(str), dist_notas.values / dist_notas.sum() * 100, color="tab:blue")
ax.set_title("Distribuição das notas (% dos pedidos avaliados)")
ax.set_xlabel("Nota")
ax.set_ylabel("%")
plt.show()

print(f"Nota média: {av_pedido['review_score'].mean():.2f}")
print(f"% de notas 1 ou 2: {(av_pedido['review_score'] <= 2).mean() * 100:.1f}%")

### Atraso na entrega vs. nota

Hipótese: pedidos entregues com atraso recebem notas piores.

In [0]:
aval_entrega = entregues.merge(av_pedido, on="order_id", how="inner", validate="one_to_one")

faixas = pd.cut(
    aval_entrega["dias_atraso"],
    bins=[-np.inf, 0, 3, 7, 14, np.inf],
    labels=["No prazo / adiantado", "1-3 dias", "4-7 dias", "8-14 dias", "Mais de 14 dias"],
)

resumo = (
    aval_entrega
    .groupby(faixas, observed=True)["review_score"]
    .agg(pedidos="count", nota_media="mean", pct_nota_baixa=lambda s: (s <= 2).mean() * 100)
)
display(resumo.round(2))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(resumo.index.astype(str), resumo["nota_media"], color="tab:red")
ax.set_ylim(1, 5)
ax.set_title("Nota média por faixa de atraso na entrega")
ax.set_ylabel("Nota média")
plt.xticks(rotation=15)
plt.show()

## 7. Distribuição geográfica

Clientes são contados por `customer_unique_id` (pessoas distintas), não por `customer_id`, que muda a cada pedido.

In [0]:
clientes_uf = dataframes["customers"].groupby("customer_state")["customer_unique_id"].nunique().rename("clientes")
vendedores_uf = dataframes["sellers"].groupby("seller_state")["seller_id"].nunique().rename("vendedores")

geo_uf = pd.concat([clientes_uf, vendedores_uf], axis=1).fillna(0).astype(int)
geo_uf["pct_clientes"] = geo_uf["clientes"] / geo_uf["clientes"].sum() * 100
geo_uf["pct_vendedores"] = geo_uf["vendedores"] / geo_uf["vendedores"].sum() * 100
geo_uf = geo_uf.sort_values("clientes", ascending=False)
display(geo_uf.head(10).round(1))

top = geo_uf.head(10)
x = np.arange(len(top))
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(x - 0.2, top["pct_clientes"], width=0.4, label="% dos clientes")
ax.bar(x + 0.2, top["pct_vendedores"], width=0.4, label="% dos vendedores")
ax.set_xticks(x)
ax.set_xticklabels(top.index)
ax.set_title("Concentração de clientes e vendedores — top 10 UFs")
ax.legend()
plt.show()

print(f"Clientes em SP: {geo_uf.loc['SP', 'pct_clientes']:.1f}% | Vendedores em SP: {geo_uf.loc['SP', 'pct_vendedores']:.1f}%")

In [0]:
geo = dataframes["geolocation"]
geo_cep = (
    geo.groupby("geolocation_zip_code_prefix")
       .agg(lat=("geolocation_lat", "median"), lng=("geolocation_lng", "median"))
       .reset_index()
)
# Remove coordenadas fora do território brasileiro
geo_cep = geo_cep[geo_cep["lat"].between(-34, 6) & geo_cep["lng"].between(-74, -34)]

pedidos_cep = (
    validos.groupby("customer_zip_code_prefix")["order_id"].nunique().rename("pedidos").reset_index()
    .merge(geo_cep, left_on="customer_zip_code_prefix", right_on="geolocation_zip_code_prefix", how="inner")
)
cobertura = pedidos_cep["pedidos"].sum() / validos["order_id"].nunique() * 100
print(f"Pedidos com coordenadas disponíveis: {cobertura:.1f}%")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(pedidos_cep["lng"], pedidos_cep["lat"], s=pedidos_cep["pedidos"].clip(upper=200) / 4,
           alpha=0.3, color="tab:blue")
ax.set_title("Pedidos por prefixo de CEP")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.show()

## 8. Recorrência de clientes

Quantos clientes compraram mais de uma vez? Aqui a chave é `customer_unique_id`.

In [0]:
pedidos_por_cliente = validos.groupby("customer_unique_id")["order_id"].nunique()
dist_rec = pedidos_por_cliente.clip(upper=4).value_counts().sort_index()
dist_rec.index = dist_rec.index.map(lambda v: "4 ou mais" if v >= 4 else str(v))

print(f"Clientes distintos: {len(pedidos_por_cliente):,}")
print(f"Clientes com mais de 1 pedido: {(pedidos_por_cliente > 1).mean() * 100:.1f}%")
display(dist_rec.rename("clientes").to_frame())